# 03 — DICOM cache builder **v4** (RSNA Knee Abnormality Detection)

Same pipeline as v3 (six slots, 24 slices, 140 mm crop, uint8, right knees mirrored,
FS-slot fallback to non-FS T2/PD, laterality by tag then geometry). The code cells equal
`src/knee/cache.py` with `CacheConfig.v4()`. What changed:

| | v3 | v4 |
|---|---|---|
| Resolution | 320 px | **384 px** (0.36 mm/px at 140 mm) |
| FS from description | any FS word counts | an FS word right after a negation ("non fatsat", "no fs", "w/o fs") does not |
| Slice order, < 80 % geometry | file name (random for SOP-UID names) | **InstanceNumber**, then file name |
| Manifest | | `order_src` per slot (geom / instance / name), `patient_id` |
| Output | `cache/` | `cache_v4/`; each npz carries `cache_key` |

**Output size.** v3 was 38 GB over 3 shards (~12.7 GB each). At 384 px a slot compresses
~1.39x larger (measured on the sample study), so each of 3 shards is **~17.5 GB against the
20 GB cap**. `N_SHARDS = 4` (~13 GB each) is the safe alternative.

**Consumers.** `slice_descriptors` (`knee.common` and its copy in the submission kernel)
reshapes slices into a 20 x 20 grid and raises on 384 px; switch it to uneven 20-cell pooling
before running the slice-direction test on v4.

## Config

In [ ]:
CFG = {
    "COMP_DIR": "/kaggle/input/rsna-knee-abnormality-detection",
    "SPLIT": "train",            # "train" or "test" (same code path)
    "CACHE_KEY": "cache_v4",     # names the output dir and every npz
    "OUT_DIR": "/kaggle/working/cache_v4",

    # Slot order is part of the cache contract: the training head indexes by position.
    "SLOTS": ["SAG_FS", "COR_FS", "AX_FS", "SAG_NOFS", "COR_T1", "SAG_T1"],
    "N_SLICES": 24,
    "BAND": (0.10, 0.90),        # fraction of the ordered stack to sample from
    "IMG": 384,
    "CROP_MM": 140.0,            # physical square centre crop before resize
    "CLIP_LO": 0.5,              # per-series percentile clip on the cropped stack -> uint8
    "CLIP_HI": 99.5,
    "MIN_SLICES": 6,             # fewer -> localizer/scout, never a slot candidate
    "LAT_DEAD_MM": 20.0,         # |centre_x| below this -> laterality unknown, no mirror
    "COMPRESS": True,
    "FS_NEGATION": True,         # a negated FS word ("non fatsat") does not count
    "INSTANCE_FALLBACK": True,   # < 80 % geometry -> InstanceNumber, then name

    "N_SHARDS": 4,               # ~17.5 GB per shard at 384 px; 4 is the safe choice
    "SHARD_INDEX": 0,

    "N_WORKERS": 0,              # 0 -> os.cpu_count()
    "TIME_BUDGET_H": 11.0,
    "SEED": 42,

    # DEBUG True -> DEBUG_N studies only.
    "DEBUG": False,
    "DEBUG_N": 20,
}
N_SLOTS = len(CFG["SLOTS"])

## Imports and setup

In [ ]:
import os
import re
import sys
import time
import zlib
import random
import traceback
import multiprocessing as mp
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from tqdm.auto import tqdm

random.seed(CFG["SEED"])
np.random.seed(CFG["SEED"])
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

try:
    import cv2
    cv2.setNumThreads(0)

    def resize(a, size):
        """Resizes one slice to size x size with OpenCV area interpolation.

        Args:
            a: 2-D float array.
            size: Output side length in pixels.

        Returns:
            The resized slice.
        """
        return cv2.resize(a, (size, size), interpolation=cv2.INTER_AREA)
except ImportError:
    from PIL import Image

    def resize(a, size):
        """Resizes one slice to size x size with PIL bilinear interpolation.

        Args:
            a: 2-D float array.
            size: Output side length in pixels.

        Returns:
            The resized slice.
        """
        return np.asarray(Image.fromarray(a).resize((size, size), Image.BILINEAR))

print("python", sys.version.split()[0], "| numpy", np.__version__, "| pydicom", pydicom.__version__)
print("cpus:", os.cpu_count())

In [ ]:
def find_root(comp_dir):
    """Locates the competition mount: a dir holding {split}.csv and {split}_series/.

    Args:
        comp_dir: The configured mount point, tried first; several Kaggle layouts and the
            local data/ directory are tried after it.

    Returns:
        The directory that holds both the split CSV and the series tree.

    Raises:
        FileNotFoundError: No candidate directory holds both.
    """
    split = CFG["SPLIT"]
    candidates = [
        Path(comp_dir),
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
        Path("data"),
        Path("."),
    ]
    def ok(c):
        """Says whether one candidate directory is the competition mount.

        Args:
            c: Candidate directory.

        Returns:
            True when it holds both {split}.csv and {split}_series/.
        """
        return (c / f"{split}.csv").is_file() and (c / f"{split}_series").is_dir()
    for c in candidates:
        if ok(c):
            return c
    base = Path("/kaggle/input")
    if base.is_dir():
        for d1 in sorted(p for p in base.iterdir() if p.is_dir()):
            for c in [d1] + sorted(p for p in d1.iterdir() if p.is_dir()):
                if ok(c):
                    return c
    raise FileNotFoundError(
        f"competition mount not found (looked at {comp_dir} and the usual alternatives)")


ROOT = find_root(CFG["COMP_DIR"])
SPLIT = CFG["SPLIT"]
SERIES_DIR = ROOT / f"{SPLIT}_series"
OUT_DIR = Path(CFG["OUT_DIR"])
OUT_DIR.mkdir(parents=True, exist_ok=True)

series_df = pd.read_csv(ROOT / f"{SPLIT}_series.csv")
STUDY_ROWS = {s: g.to_dict("records") for s, g in series_df.groupby("StudyInstanceUID")}
print("root:", ROOT)
print("series rows:", len(series_df), "| studies:", len(STUDY_ROWS))

## Header parsing: contrast, fat suppression, geometry, laterality

One header read per series (the middle file, `stop_before_pixels`). The delivered
`Fluid_Sensitive` flag equals `Fat_Suppression` on every row, so it is used as the FS
flag; T1 vs PD vs T2 has to come from the header. Precedence: SeriesDescription words,
then TR/TE (TR < 800 ms → T1; TE > 60 ms → T2; else PD), with STIR counted as FS.
v4 (`FS_NEGATION`): an FS word directly after "non", "no", "not", "without" or "w/o" does
not count; the csv flag and ScanOptions still add FS as before.

In [ ]:
HDR_TAGS = [
    "SeriesDescription", "ProtocolName", "SequenceName", "ScanningSequence", "ScanOptions",
    "RepetitionTime", "EchoTime", "InversionTime", "Laterality", "ImageLaterality",
    "PixelSpacing", "Rows", "Columns", "ImageOrientationPatient", "ImagePositionPatient",
    "PhotometricInterpretation", "SeriesInstanceUID", "PatientID",
]
_RX_LOC = re.compile(r"loc|scout|survey|plan|calib|smartbrain|3.?plane", re.I)
_RX_T1 = re.compile(r"\bt1\b|t1w|t1_|_t1|t1 ", re.I)
_RX_T2 = re.compile(r"\bt2\b|t2w|t2_|_t2|t2 ", re.I)
_RX_PD = re.compile(r"\bpd\b|\bdp\b|proton|dens|pdw", re.I)
_RX_STIR = re.compile(r"stir|tirm", re.I)
_RX_FS = re.compile(r"\bfs\b|fat.?sat|spir|spair|fatsup|\bsat\b|fs_|_fs|fs ", re.I)
# Cache v4: the v3 words plus "fat sup(pressed)" and STIR/TIRM. A mention does not count when
# a negation word sits right before it, glued or after a separator ("nonfs", "no_fs", "w/o fs").
_RX_FS_V4 = re.compile(_RX_FS.pattern + r"|fat.?sup|stir|tirm", re.I)
_RX_FS_NEG = re.compile(r"(?<![a-z0-9])(?:non|no|not|without|w/o|w/out|wo)[\s_\-./]*$", re.I)


def _f(x, default=None):
    """Coerces a DICOM value to float.

    Args:
        x: Any value, typically a pydicom element value.
        default: What to return when x is not numeric.

    Returns:
        The float, or default.
    """
    try:
        return float(x)
    except (TypeError, ValueError):
        return default


def read_header(series_path):
    """Parses one series' header into a plain dict. Never raises.

    The middle file of the series is read with stop_before_pixels, so one header read
    stands for the whole series.

    Args:
        series_path: Directory holding the series' .dcm files.

    Returns:
        A dict with n_files and hdr_ok always present, and, when the header could be read,
        desc, scan_seq, scan_opts, tr, te, ti, lat_tag, patient_id, ps_row, ps_col, rows,
        cols, iop, ipp and centre_x. hdr_ok is 0 when only the file count is known.
    """
    files = sorted(p for p in Path(series_path).iterdir() if p.name.lower().endswith(".dcm"))
    h = {"n_files": len(files), "hdr_ok": 0}
    if not files:
        return h
    try:
        ds = pydicom.dcmread(str(files[len(files) // 2]), stop_before_pixels=True,
                             force=True, specific_tags=HDR_TAGS)
    except (pydicom.errors.InvalidDicomError, OSError, ValueError, KeyError, AttributeError):
        return h
    h["hdr_ok"] = 1
    h["desc"] = " ".join(str(getattr(ds, t, "") or "") for t in
                         ("SeriesDescription", "ProtocolName", "SequenceName"))
    h["scan_seq"] = str(getattr(ds, "ScanningSequence", "") or "")
    h["scan_opts"] = str(getattr(ds, "ScanOptions", "") or "")
    h["tr"] = _f(getattr(ds, "RepetitionTime", None))
    h["te"] = _f(getattr(ds, "EchoTime", None))
    h["ti"] = _f(getattr(ds, "InversionTime", None))
    lat = str(getattr(ds, "ImageLaterality", "")
              or getattr(ds, "Laterality", "") or "").strip().upper()
    h["lat_tag"] = lat if lat in ("L", "R") else ""
    h["patient_id"] = str(getattr(ds, "PatientID", "") or "").strip()
    ps = getattr(ds, "PixelSpacing", None)
    h["ps_row"] = _f(ps[0]) if ps is not None and len(ps) >= 2 else None
    h["ps_col"] = _f(ps[1]) if ps is not None and len(ps) >= 2 else None
    h["rows"] = int(getattr(ds, "Rows", 0) or 0)
    h["cols"] = int(getattr(ds, "Columns", 0) or 0)
    iop = getattr(ds, "ImageOrientationPatient", None)
    ipp = getattr(ds, "ImagePositionPatient", None)
    h["iop"] = [float(x) for x in iop[:6]] if iop is not None and len(iop) >= 6 else None
    h["ipp"] = [float(x) for x in ipp[:3]] if ipp is not None and len(ipp) >= 3 else None
    h["centre_x"] = image_centre_x(h)
    return h


def image_centre_x(h):
    """Patient-x (LPS: +x = patient's left) of the image centre, in mm.

    Args:
        h: A header dict from read_header; iop, ipp, ps_row/ps_col and rows/cols are read.

    Returns:
        The x coordinate in mm, or None when the geometry tags are missing.
    """
    if not h.get("iop") or not h.get("ipp") or not h.get("ps_row") or not h.get("rows"):
        return None
    r = np.asarray(h["iop"][:3])   # direction of increasing column index
    c = np.asarray(h["iop"][3:])   # direction of increasing row index
    centre = (np.asarray(h["ipp"]) + r * (h["cols"] - 1) / 2 * h["ps_col"]
              + c * (h["rows"] - 1) / 2 * h["ps_row"])
    return float(centre[0])


def fs_in_description(desc, ignore_negated=False):
    """Says whether a series description names fat suppression.

    Args:
        desc: SeriesDescription, ProtocolName and SequenceName joined by spaces.
        ignore_negated: Use the v4 test: the v3 words plus "fat sup(pressed)", STIR and TIRM,
            where a mention directly preceded by a negation word ("non fatsat", "no_fs",
            "w/o fs", "nonfs") does not count. False is the v3 test, where every mention
            counts.

    Returns:
        True when at least one mention counts.
    """
    if not ignore_negated:
        return bool(_RX_FS.search(desc))
    return any(not _RX_FS_NEG.search(desc[:m.start()]) for m in _RX_FS_V4.finditer(desc))


def classify_contrast(h, csv_fs, fs_negation=False):
    """Decides a series' tissue contrast and whether it is fat-suppressed.

    Precedence: SeriesDescription words, then TR/TE (TR < 800 ms -> T1; TE > 60 ms -> T2;
    else PD), with STIR counted as fat-suppressed whatever the other flags say.

    Args:
        h: A header dict from read_header.
        csv_fs: The delivered Fluid_Sensitive flag for this series; truthy forces fs.
        fs_negation: Whether a negated FS mention in the description is ignored (v4).

    Returns:
        The contrast, one of {T1, PD, T2, STIR, GRE, UNK}, and the fat-suppression flag.
    """
    desc = h.get("desc", "") or ""
    opts = {t.strip().upper() for t in re.split(r"[\\/,;]", h.get("scan_opts") or "")}
    fs = (bool(csv_fs) or fs_in_description(desc, fs_negation)
          or bool(opts & {"FS", "FAT_SAT", "FATSAT", "SPIR", "SPAIR"}))
    if _RX_STIR.search(desc) or (h.get("ti") and 80 <= h["ti"] <= 250):
        return "STIR", True
    if _RX_T1.search(desc):
        return "T1", fs
    if _RX_T2.search(desc):
        return "T2", fs
    if _RX_PD.search(desc):
        return "PD", fs
    tr, te = h.get("tr"), h.get("te")
    if tr is not None and te is not None:
        if tr < 800:
            return "T1", fs
        if te > 60:
            return "T2", fs
        return "PD", fs
    if "GR" in (h.get("scan_seq") or ""):
        return "GRE", fs
    return "UNK", fs


def is_localizer(h):
    """Says whether a series is a localizer/scout rather than a diagnostic acquisition.

    Args:
        h: A header dict from read_header.

    Returns:
        True when the series has fewer than MIN_SLICES files or its description names a
        localizer, survey, plan or calibration scan.
    """
    return h.get("n_files", 0) < CFG["MIN_SLICES"] or bool(_RX_LOC.search(h.get("desc", "") or ""))

## Slot selection

Each slot has a (plane, fs, contrast) requirement. Candidates are ranked by slice count
(more slices = the diagnostic acquisition, not a reformat), UID as the tie-break. A series
fills at most one slot; slots are filled in the order listed so the fluid-sensitive
workhorses get first pick.

In [ ]:
SLOT_SPEC = {
    "SAG_FS":   ("Sagittal", True,  {"PD", "T2", "STIR", "UNK", "GRE"}),
    "COR_FS":   ("Coronal",  True,  {"PD", "T2", "STIR", "UNK", "GRE"}),
    "AX_FS":    ("Axial",    True,  {"PD", "T2", "STIR", "UNK", "GRE"}),
    "SAG_NOFS": ("Sagittal", False, {"PD", "T2", "GRE", "UNK"}),
    "COR_T1":   ("Coronal",  None,  {"T1"}),
    "SAG_T1":   ("Sagittal", None,  {"T1"}),
}


# For an FS slot with no FS series: the next-most fluid-sensitive contrast, in order.
FALLBACK_ORDER = ["T2", "PD", "GRE", "UNK"]


def pick_slots(meta):
    """Chooses at most one series per cache slot.

    Pass 1 fills every slot from series that match its spec exactly. Pass 2 fills any FS slot
    still empty with a leftover same-plane non-FS T2, then PD (~6 % of studies have no
    fat-suppressed sagittal at all; their T2 is fluid-sensitive and otherwise unused).
    A series never fills two slots. Candidates are ranked by slice count, UID as the
    tie-break, so the result does not depend on the order `meta` arrives in.

    Args:
        meta: One dict per series with uid, plane, fs, contrast, n_files and localizer.

    Returns:
        Two lists of length N_SLOTS in CFG["SLOTS"] order: the chosen series UID per slot
        (None where the slot stays empty), and whether that slot was filled by the pass-2
        fallback.
    """
    used, chosen, fallback = set(), [], [False] * N_SLOTS
    for name in CFG["SLOTS"]:
        plane, want_fs, contrasts = SLOT_SPEC[name]
        cands = [m for m in meta
                 if m["uid"] not in used and not m["localizer"] and m["plane"] == plane
                 and m["contrast"] in contrasts
                 and (want_fs is None or bool(m["fs"]) == want_fs)]
        cands.sort(key=lambda m: (-m["n_files"], m["uid"]))
        pick = cands[0]["uid"] if cands else None
        if pick is not None:
            used.add(pick)
        chosen.append(pick)
    for i, name in enumerate(CFG["SLOTS"]):
        plane, want_fs, _ = SLOT_SPEC[name]
        if chosen[i] is not None or want_fs is not True:
            continue
        for con in FALLBACK_ORDER:
            cands = [m for m in meta if m["uid"] not in used and not m["localizer"]
                     and m["plane"] == plane and m["contrast"] == con]
            if cands:
                cands.sort(key=lambda m: (-m["n_files"], m["uid"]))
                chosen[i] = cands[0]["uid"]
                used.add(chosen[i])
                fallback[i] = True
                break
    return chosen, fallback


def study_laterality(meta):
    """Decides which knee a study images.

    The Laterality/ImageLaterality tag wins when any series carries one; otherwise the
    median image-centre x decides, unless it falls inside the dead zone around the midline.

    Args:
        meta: One dict per series, with lat_tag and centre_x.

    Returns:
        The side, one of {'L', 'R', 'U'}, and where it came from: 'tag', 'geom',
        'geom_dead:<x>' or 'none'.
    """
    tags = [m["lat_tag"] for m in meta if m.get("lat_tag")]
    if tags:
        side = max(("L", "R"), key=tags.count)
        return side, "tag"
    xs = [m["centre_x"] for m in meta if m.get("centre_x") is not None]
    if xs:
        x = float(np.median(xs))
        if abs(x) >= CFG["LAT_DEAD_MM"]:
            return ("L" if x > 0 else "R"), "geom"
        return "U", f"geom_dead:{x:.0f}"
    return "U", "none"


def study_patient_id(meta):
    """Picks the study's PatientID from its series headers.

    Args:
        meta: One dict per series, with patient_id ('' when the tag was absent).

    Returns:
        The most frequent non-empty PatientID, the alphabetically first on a tie, or ''
        when no series carries one.
    """
    ids = [m["patient_id"] for m in meta if m.get("patient_id")]
    return max(sorted(set(ids)), key=ids.count) if ids else ""

## Pixel loading, canonical orientation, physical crop

Slices are ordered by projecting `ImagePositionPatient` onto the slice normal; when under
80 % of files carry it, v4 orders by `InstanceNumber`, then file name (`order_src`). The volume
is then flipped so that every axis increases toward +x (left), +y (posterior) or +z
(superior), whichever the axis is aligned with. For a **right** knee the axis aligned
with x is mirrored, so medial/lateral mean the same image direction for every study.
The crop is a fixed 140 mm square around the image centre (zero-padded when the FOV is
smaller), so one pixel is the same physical size for every series.

In [ ]:
ORDER_TAGS = ["ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"]


def order_files(series_path, instance_fallback=False):
    """Sorts a series' files into through-plane order.

    Each file is keyed by projecting ImagePositionPatient onto the slice normal, falling
    back to InstanceNumber and then to the file's position in the directory listing. A
    part-geometric sort is worse than none, so when fewer than 80 % of files carry usable
    geometry v3 keeps the plain name order, which is random for SOP-UID file names. With
    instance_fallback the files are then sorted by InstanceNumber when at least 80 % carry
    one, and only otherwise left in name order.

    Args:
        series_path: Directory holding the series' .dcm files.
        instance_fallback: Try InstanceNumber order before name order (v4).

    Returns:
        The ordered paths; the slice normal as an unquantised vector, or None when the
        order is not geometric; and where the order came from: 'geom', 'instance' or
        'name' ('' when the directory holds no .dcm file).
    """
    files = sorted(p for p in Path(series_path).iterdir() if p.name.lower().endswith(".dcm"))
    if not files:
        return [], None, ""
    geom, inst, normal = [], [], None
    for f in files:
        g = k = None
        try:
            ds = pydicom.dcmread(str(f), stop_before_pixels=True, force=True,
                                 specific_tags=ORDER_TAGS)
            ipp = getattr(ds, "ImagePositionPatient", None)
            iop = getattr(ds, "ImageOrientationPatient", None)
            if ipp is not None and iop is not None and len(ipp) >= 3 and len(iop) >= 6:
                p = np.asarray([float(x) for x in ipp[:3]])
                o = np.asarray([float(x) for x in iop[:6]])
                n = np.cross(o[:3], o[3:])
                d = float(np.dot(p, n))
                if np.isfinite(d):
                    g, normal = d, n
            # v3 reads InstanceNumber only where geometry is missing; keeping that means a
            # malformed tag on a geometric file cannot change its v3 key.
            if g is None or instance_fallback:
                num = getattr(ds, "InstanceNumber", None)
                if num is not None:
                    k = float(num)
        except (pydicom.errors.InvalidDicomError, OSError, ValueError, KeyError,
                AttributeError, TypeError):
            pass
        geom.append(g)
        inst.append(k)
    n_geom = sum(g is not None for g in geom)
    n_inst = sum(k is not None for k in inst)
    if n_geom >= 0.8 * len(files) or (n_geom == 0 and not instance_fallback):
        keys = [g if g is not None else k if k is not None else float(i)
                for i, (g, k) in enumerate(zip(geom, inst))]
        order = [f for _, f in sorted(zip(keys, files), key=lambda t: t[0])]
        return order, normal, "geom" if n_geom else ("instance" if n_inst else "name")
    if instance_fallback and n_inst >= 0.8 * len(files):
        idx = sorted(range(len(files)), key=lambda i: (inst[i] is None, inst[i] or 0.0, i))
        return [files[i] for i in idx], None, "instance"
    return files, None, "name"


def sample_indices(n, k, band):
    """Picks k evenly spaced slice indices from the central band of a stack.

    Args:
        n: How many slices the series has.
        k: How many to sample; indices repeat when k exceeds the band's width.
        band: (lo, hi) fractions of the ordered stack to sample between, inclusive.

    Returns:
        k integer indices, ascending, all within [0, n - 1]. All zeros when n <= 0.
    """
    if n <= 0:
        return np.zeros(k, dtype=int)
    lo, hi = band
    return np.linspace(lo * (n - 1), hi * (n - 1), k).round().astype(int)


def read_pixels(path):
    """Reads one DICOM file's pixels as a rescaled 2-D float array.

    Args:
        path: The .dcm file.

    Returns:
        The slice after RescaleSlope/Intercept, negated when the photometric
        interpretation is MONOCHROME1 so that bright always means high signal, or None
        when the file could not be decoded.
    """
    try:
        ds = pydicom.dcmread(str(path), force=True)
        a = ds.pixel_array.astype(np.float32)
        if a.ndim == 3:
            a = a[0] if a.shape[0] < a.shape[-1] else a[..., 0]
        a = (a * float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
             + float(getattr(ds, "RescaleIntercept", 0.0) or 0.0))
        if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
            a = -a
        return a
    # Pixel decoders raise a wide range of types (unsupported transfer syntax, truncated
    # file, missing element). One bad slice must never fail the study, so the list is broad,
    # but it is still narrower than `except Exception`.
    except (pydicom.errors.InvalidDicomError, OSError, ValueError, KeyError, AttributeError,
            TypeError, NotImplementedError, RuntimeError):
        return None


def crop_mm(a, ps_row, ps_col, size_mm):
    """Crops a slice to a fixed physical square around its centre.

    One output pixel is then the same physical size for every series. When the field of
    view is smaller than the requested square, the margin is padded with the slice's own
    minimum value (not zero), so the pad matches the image's background level.

    Args:
        a: The 2-D slice.
        ps_row: Row pixel spacing in mm; a falsy value returns `a` unchanged.
        ps_col: Column pixel spacing in mm; a falsy value returns `a` unchanged.
        size_mm: Side of the square crop in mm.

    Returns:
        A float32 array of round(size_mm / ps_row) x round(size_mm / ps_col) pixels, or
        `a` itself when the pixel spacing is unknown.
    """
    if not ps_row or not ps_col:
        return a
    hr, hc = int(round(size_mm / ps_row)), int(round(size_mm / ps_col))
    n_rows, n_cols = a.shape
    r0, c0 = (n_rows - hr) // 2, (n_cols - hc) // 2
    out = np.full((hr, hc), float(a.min()), dtype=np.float32)
    sr0, sc0 = max(r0, 0), max(c0, 0)
    sr1, sc1 = min(r0 + hr, n_rows), min(c0 + hc, n_cols)
    out[sr0 - r0: sr1 - r0, sc0 - c0: sc1 - c0] = a[sr0:sr1, sc0:sc1]
    return out


CANON_SIGN = {0: 1.0, 1: 1.0, 2: -1.0}   # LPS: +x left, +y posterior, -z inferior (display order)


def canonical_flips(iop, normal, side):
    """Lists which axes of (slice, row, col) to flip for canonical orientation.

    Canonical is the standard radiological display: columns run right->left or
    anterior->posterior, rows run superior->inferior or anterior->posterior, and slices
    ascend toward +x/+y/-z. A right knee is then mirrored along whichever axis is aligned
    with patient x, so medial and lateral mean the same image direction for every study.

    Args:
        iop: ImageOrientationPatient as six floats (column direction then row direction),
            or None when the tag is missing.
        normal: The slice normal, or None when no geometry was recovered.
        side: 'L', 'R' or 'U' from study_laterality.

    Returns:
        The axes to flip, a subset of {0: slice, 1: row, 2: col}. Empty when iop is None.
    """
    if iop is None:
        return []
    col_dir = np.asarray(iop[:3])   # increasing column index
    row_dir = np.asarray(iop[3:])   # increasing row index
    dirs = {0: normal, 1: row_dir, 2: col_dir}
    flips = []
    for ax, d in dirs.items():
        if d is None:
            continue
        dom = int(np.argmax(np.abs(d)))
        if d[dom] * CANON_SIGN[dom] < 0:
            flips.append(ax)
        if side == "R" and dom == 0:        # x-aligned axis: mirror right knees
            flips = [f for f in flips if f != ax] if ax in flips else flips + [ax]
    return flips


def load_series(series_path, h, side, n_slices, img, band, crop, clip_lo, clip_hi,
                instance_fallback=False):
    """Loads one series into the cache's uint8 slot format.

    Slices are ordered, sampled over the central band, physically cropped, resized,
    flipped into canonical orientation and finally scaled by a percentile clip taken over
    the whole sampled stack. A slice that fails to decode is replaced by the previous good
    one, so the slot always has n_slices planes.

    Args:
        series_path: Directory holding the series' .dcm files.
        h: The series' header dict, read for pixel spacing and orientation.
        side: The study's laterality, for the right-knee mirror.
        n_slices: Slices to store.
        img: Output side length in pixels.
        band: (lo, hi) fractions of the stack to sample between.
        crop: Physical crop size in mm.
        clip_lo: Lower percentile of the clip, over the whole sampled stack.
        clip_hi: Upper percentile of the clip.
        instance_fallback: Passed to order_files.

    Returns:
        A (n_slices, img, img) uint8 array, or None when the series yielded nothing; a
        note, one of 'ok', 'partial:<n>' with the count of substituted slices, 'no_dcm' or
        'all_slices_failed'; and the slice-order source from order_files.
    """
    files, normal, order_src = order_files(series_path, instance_fallback)
    if not files:
        return None, "no_dcm", order_src
    idx = sample_indices(len(files), n_slices, band)
    planes, bad = [], 0
    for i in idx:
        a = read_pixels(files[int(i)])
        if a is None:
            bad += 1
            planes.append(None)
            continue
        cropped = crop_mm(a, h.get("ps_row"), h.get("ps_col"), crop)
        planes.append(resize(cropped, img).astype(np.float32))
    good = [p for p in planes if p is not None]
    if not good:
        return None, "all_slices_failed", order_src
    last = None
    for j, p in enumerate(planes):
        if p is None:
            planes[j] = last if last is not None else good[0]
        else:
            last = p
    vol = np.stack(planes, axis=0)
    for ax in canonical_flips(h.get("iop"), normal, side):
        vol = np.flip(vol, axis=ax)
    lo, hi = np.percentile(vol, (clip_lo, clip_hi))
    vol = np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1)
    return (np.ascontiguousarray((vol * 255.0).round().astype(np.uint8)),
            f"partial:{bad}" if bad else "ok", order_src)

## Per-study worker

In [ ]:
W = dict(CFG)
W["SERIES_DIR"] = str(SERIES_DIR)
W["OUT_DIR"] = str(OUT_DIR)


def out_path(study):
    """Gives the cache file path for one study.

    Args:
        study: StudyInstanceUID.

    Returns:
        The .npz path under the configured output directory.
    """
    return Path(W["OUT_DIR"]) / f"{study}.npz"


def process_study(study):
    """Builds one study's cache file. Never raises.

    Written atomically: the array goes to a .tmp.npz that is renamed into place, so an
    interrupted shard leaves no half-written file for the resume pass to trust.

    Args:
        study: StudyInstanceUID.

    Returns:
        A manifest record and one metadata row per series. The record's status is 'ok',
        'skip' when the file already exists, 'empty' when no slot could be filled, or
        'fail' with the traceback in its note. Its order_src holds the slice-order source
        per slot joined by '|' ('geom', 'instance' or 'name'; empty for an empty slot), and
        its patient_id the study's PatientID ('' when absent).
    """
    dst = out_path(study)
    rec = {"StudyInstanceUID": study, "status": "ok", "slots": "", "side": "",
           "side_src": "", "fallback": "", "order_src": "", "patient_id": "", "note": ""}
    meta_rows = []
    try:
        if dst.is_file() and dst.stat().st_size > 0:
            rec["status"] = "skip"
            return rec, meta_rows

        meta = []
        for r in STUDY_ROWS.get(study, []):
            uid = str(r["SeriesInstanceUID"])
            path = Path(W["SERIES_DIR"]) / str(study) / uid
            h = read_header(path) if path.is_dir() else {"n_files": 0, "hdr_ok": 0}
            contrast, fs = classify_contrast(h, r.get("Fluid_Sensitive", 0), W["FS_NEGATION"])
            m = {"uid": uid, "plane": str(r.get("Anatomical_Plane", "")), "fs": fs,
                 "contrast": contrast, "n_files": h.get("n_files", 0),
                 "localizer": is_localizer(h), "lat_tag": h.get("lat_tag", ""),
                 "centre_x": h.get("centre_x"), "patient_id": h.get("patient_id", ""), "h": h}
            meta.append(m)
            meta_rows.append({"StudyInstanceUID": study, "SeriesInstanceUID": uid,
                              "plane": m["plane"],
                              "csv_fs": int(r.get("Fluid_Sensitive", 0) or 0),
                              "fs": int(fs), "contrast": contrast,
                              "n_files": m["n_files"], "localizer": int(m["localizer"]),
                              "tr": h.get("tr"), "te": h.get("te"), "lat_tag": m["lat_tag"],
                              "centre_x": m["centre_x"], "ps_row": h.get("ps_row"),
                              "patient_id": m["patient_id"],
                              "desc": (h.get("desc") or "")[:80]})

        rec["patient_id"] = study_patient_id(meta)
        side, side_src = study_laterality(meta)
        picks, fallback = pick_slots(meta)
        by_uid = {m["uid"]: m for m in meta}

        arr = np.zeros((N_SLOTS, W["N_SLICES"], W["IMG"], W["IMG"]), dtype=np.uint8)
        present = np.zeros(N_SLOTS, dtype=np.uint8)
        notes, order_src = [], [""] * N_SLOTS
        for s, uid in enumerate(picks):
            if uid is None:
                continue
            path = Path(W["SERIES_DIR"]) / str(study) / uid
            try:
                vol, note, src = load_series(path, by_uid[uid]["h"], side, W["N_SLICES"],
                                             W["IMG"], W["BAND"], W["CROP_MM"], W["CLIP_LO"],
                                             W["CLIP_HI"], W["INSTANCE_FALLBACK"])
            # Deliberate isolation point: one unreadable series must cost its slot, not the
            # study. The exception type is recorded in the manifest note.
            except Exception as e:
                vol, note = None, f"err:{type(e).__name__}"
            if vol is None:
                notes.append(f"{s}:{note}")
                continue
            arr[s] = vol
            present[s] = 1
            order_src[s] = src
            if note != "ok":
                notes.append(f"{s}:{note}")

        if present.sum() == 0:
            rec.update(status="empty", note=";".join(notes))
            return rec, meta_rows

        tmp = dst.with_suffix(".tmp.npz")
        save = np.savez_compressed if W["COMPRESS"] else np.savez
        key = {} if W["CACHE_KEY"] == "cache_v3" else {"cache_key": np.array(W["CACHE_KEY"])}
        save(tmp, vol=arr, present=present, side=np.array(side),
             slot_uids=np.array([u or "" for u in picks]),
             slot_fallback=np.array(fallback, dtype=np.uint8), **key)
        os.replace(tmp, dst)
        rec.update(slots="".join(str(int(x)) for x in present), side=side, side_src=side_src,
                   fallback="".join(str(int(x)) for x in fallback),
                   order_src="|".join(order_src), note=";".join(notes))
        return rec, meta_rows
    # Deliberate isolation point: this is the whole body of a pool worker, and a shard of
    # 1500 studies must not die on one of them. The traceback goes into the manifest.
    except Exception:
        rec.update(status="fail", note=traceback.format_exc(limit=2).replace("\n", " | "))
        return rec, meta_rows

## Run

In [ ]:
studies = sorted(STUDY_ROWS)
if CFG["N_SHARDS"] > 1:
    studies = [s for s in studies if zlib.crc32(s.encode()) % CFG["N_SHARDS"] == CFG["SHARD_INDEX"]]
    print(f"shard {CFG['SHARD_INDEX']}/{CFG['N_SHARDS']}: {len(studies)} studies")
if CFG["DEBUG"]:
    studies = studies[: CFG["DEBUG_N"]]
    print(f"DEBUG: {len(studies)} studies only")
todo = [s for s in studies if not (out_path(s).is_file() and out_path(s).stat().st_size > 0)]
print(f"{len(studies)} in scope | {len(studies) - len(todo)} cached | {len(todo)} to build")

In [ ]:
n_workers = CFG["N_WORKERS"] or os.cpu_count() or 2
budget = CFG["TIME_BUDGET_H"] * 3600
t0 = time.time()
records, meta_all, counts = [], [], {}

if todo:
    ctx = mp.get_context("fork")
    with ctx.Pool(n_workers) as pool:
        it = pool.imap_unordered(process_study, todo, chunksize=1)
        bar = tqdm(it, total=len(todo), desc="studies", smoothing=0.02)
        for i, (rec, mrows) in enumerate(bar, 1):
            records.append(rec)
            meta_all.extend(mrows)
            counts[rec["status"]] = counts.get(rec["status"], 0) + 1
            elapsed = time.time() - t0
            rate = elapsed / i
            bar.set_postfix(ok=counts.get("ok", 0), bad=len(records) - counts.get("ok", 0),
                            eta_h=f"{rate * (len(todo) - i) / 3600:.2f}")
            if i in (25, 100) or i % 500 == 0:
                print(f"[{elapsed/60:6.1f} min] {i}/{len(todo)} | {rate:.2f} s/study | "
                      f"projected total {rate * len(todo) / 3600:.2f} h | {counts}")
            if elapsed > budget:
                print(f"\ntime budget reached at {i}/{len(todo)}; rerun to resume.")
                pool.terminate()
                break
print(f"\ndone in {(time.time() - t0)/60:.1f} min | {counts}")

## Manifest, series metadata, coverage checks

In [ ]:
tag = f"{SPLIT}_s{CFG['SHARD_INDEX']}"
man_path = OUT_DIR.parent / f"cache_manifest_{tag}.csv"
meta_path = OUT_DIR.parent / f"series_meta_{tag}.csv"

new = pd.DataFrame(records) if records else pd.DataFrame(
    columns=["StudyInstanceUID", "status", "slots", "side", "side_src", "fallback",
             "order_src", "patient_id", "note"])
if man_path.is_file():
    # Read patient_id back as text: an all-digit ID must not become a number.
    old = pd.read_csv(man_path, dtype={"patient_id": str})
    new = (pd.concat([old, new], ignore_index=True)
             .drop_duplicates("StudyInstanceUID", keep="last"))
new.to_csv(man_path, index=False)

meta_df = pd.DataFrame(meta_all)
if meta_path.is_file() and len(meta_df):
    meta_df = (pd.concat([pd.read_csv(meta_path), meta_df], ignore_index=True)
                 .drop_duplicates("SeriesInstanceUID", keep="last"))
if len(meta_df):
    meta_df.to_csv(meta_path, index=False)

files = sorted(OUT_DIR.glob("*.npz"))
size_gb = sum(f.stat().st_size for f in files) / 1e9
print(f"cache files: {len(files)} | {size_gb:.2f} GB | "
      f"{size_gb / max(len(files), 1) * 1000:.1f} MB per study")
if files and len(files) < len(studies):
    print(f"projected shard size: {size_gb / len(files) * len(studies):.1f} GB")
print(new["status"].value_counts().to_string())

done = new[new.status == "ok"]
if len(done):
    # zfill: a manifest read back on resume has lost the string's leading zeros to the CSV.
    cov = np.array([[int(c) for c in s] for s in done.slots.astype(str).str.zfill(N_SLOTS)])
    print("\nslot coverage (fraction of studies with the slot filled):")
    for name, c in zip(CFG["SLOTS"], cov.mean(axis=0)):
        print(f"  {name:9s} {c:.3f}")
    print("\nlaterality:", done.side.value_counts().to_dict(),
          "| source:", done.side_src.str.split(":").str[0].value_counts().to_dict())
    fb = np.array([[int(c) for c in s] for s in done.fallback.astype(str).str.zfill(N_SLOTS)])
    print("fallback-filled fraction per slot:", dict(zip(CFG["SLOTS"], fb.mean(axis=0).round(3))))
    if "order_src" in done:
        src = done.order_src.fillna("").astype(str).str.split("|").explode()
        print("slice order source (filled slots):", src[src != ""].value_counts().to_dict())
if "patient_id" in new:
    pid = new.patient_id.fillna("").astype(str)
    per_patient = pid[pid != ""].value_counts()
    print(f"patient_id: {int(per_patient.sum())} studies carry one | {len(per_patient)} patients | "
          f"{int((per_patient > 1).sum())} with several studies")
if len(meta_df):
    print("\ncontrast x fs x plane (all series seen):")
    print(pd.crosstab([meta_df.plane, meta_df.contrast], meta_df.fs).to_string())
    both = meta_df[(meta_df.lat_tag != "") & meta_df.centre_x.notna()]
    if len(both):
        geom = np.where(both.centre_x > 0, "L", "R")
        far = both.centre_x.abs() >= CFG["LAT_DEAD_MM"]
        agree = (geom[far] == both.lat_tag[far]).mean()
        print(f"\ntag vs geometry agreement (|x| >= dead zone): "
              f"{agree:.3f} on {far.sum()} series")
bad = new[~new.status.isin(["ok", "skip"])]
if len(bad):
    print("\nfailures (first 10):")
    print(bad.head(10).to_string(index=False))

In [ ]:
# Eyeball one study: every slot, four depths. A cache nobody looked at has a bug in it.
if files:
    d = np.load(files[0])
    vol, present = d["vol"], d["present"]
    print(files[0].name, "| side", d["side"], "| present", present, "| shape", vol.shape,
          "| key", d["cache_key"] if "cache_key" in d else "cache_v3")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(N_SLOTS, 4, figsize=(10, 2.4 * N_SLOTS))
    for s in range(N_SLOTS):
        for j, k in enumerate(np.linspace(2, CFG["N_SLICES"] - 3, 4).astype(int)):
            ax[s, j].imshow(vol[s, k], cmap="gray", vmin=0, vmax=255)
            ax[s, j].set_axis_off()
            ax[s, j].set_title(f"{CFG['SLOTS'][s]} / {k}"
                               + ("" if present[s] else " (absent)"), fontsize=8)
    plt.tight_layout()
    plt.show()

## How to run

1. DEBUG run (20 studies): check slot coverage, `order_src`, `patient_id`, the contrast
   crosstab and the picture, and read the projected shard size against 20 GB.
2. Set `DEBUG=False`; push once per `SHARD_INDEX` in {0, 1, 2} as kernels
   `rsna-knee-cache-v4-s0/s1/s2`. Each writes `/kaggle/working/cache_v4` (~17.5 GB). If the
   DEBUG projection is above ~18 GB, use `N_SHARDS=4` and four kernels instead.
3. Train with `CACHE_KEY = "rsna-knee-cache-v4"` (the loader prefers paths containing it).
   The submission kernel must rebuild the test cache with these v4 cells.